<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# C18's Model, Piloted

A pilot, not a test: C18's plan (STATE.md, *Blocked*) was parked because its model, run for a fixed 20 steps, would
refute the hunch by construction. This notebook runs the model to a steady state, sweeps the mean degree, and measures
how wide the transition is at three sizes, so that the plan's margins can be written from what the model does. It
decides nothing about the hunch.

The model: N units, directed random edges with mean degree d (Poisson); the seed unit held at 1; each step a unit's
activation becomes half its last value plus its weighted input, clipped to [0, 1]; run until no unit moves by more
than 1e-10 (at most 2000 steps). A unit is activated above 0.1. Variants: **pure**, every edge 0.2; **plan budget**,
the plan's version, each unit's output 0.2 x activation x the network's mean degree split among its targets;
**true budget**, each unit's output c x activation split among its targets whatever the degree, for c = 0.2 and 0.6;
**fan-out**, edges 0.2 / ln(1 + out-degree); **inhibition**, the pure input minus half the mean activation of a unit's
inputs. Linear reasoning, not a result: the pure network grows from the seed once 0.2 d / 0.5 > 1, so near d = 2.5.

In [1]:
import numpy as np

def network(n, d, rng):
    m = rng.poisson(d * n)
    src, dst = rng.integers(0, n, m), rng.integers(0, n, m)
    keep = src != dst
    return src[keep], dst[keep]

def steady(n, src, dst, model, steps=2000, tol=1e-10):
    outdeg = np.bincount(src, minlength=n)
    indeg = np.bincount(dst, minlength=n)
    if model in ("pure", "inhibition"):
        w = np.full(src.size, 0.2)
    elif model == "plan budget":
        w = 0.2 * (src.size / n) / outdeg[src]
    elif model.startswith("true budget"):
        w = float(model.split()[-1]) / outdeg[src]
    elif model == "fan-out":
        w = 0.2 / np.log1p(outdeg[src])
    a = np.zeros(n); a[0] = 1.0
    for _ in range(steps):
        drive = np.bincount(dst, weights=w * a[src], minlength=n)
        if model == "inhibition":
            drive = drive - 0.5 * np.bincount(dst, weights=a[src], minlength=n) / np.maximum(indeg, 1)
        new = np.clip(0.5 * a + drive, 0.0, 1.0)
        new[0] = 1.0
        done = np.max(np.abs(new - a)) < tol
        a = new
        if done:
            break
    return np.mean(a[1:] > 0.1)

MODELS = ["pure", "plan budget", "true budget 0.2", "true budget 0.6", "fan-out", "inhibition"]
SIZES, DEGREES, DRAWS = [100, 1000, 10000], np.arange(0.5, 6.01, 0.25), 10
rng = np.random.default_rng(20261014)
curves = {}
for n in SIZES:
    for model in MODELS:
        curves[(n, model)] = np.array([[steady(n, *network(n, d, rng), model) for _ in range(DRAWS)] for d in DEGREES])
print("numpy", np.__version__, "seed 20261014,", DRAWS, "draws per point")

numpy 2.5.4 seed 20261014, 10 draws per point


In [2]:
def width(mean):
    """The degrees at which the mean activated fraction first passes 10% and 90% of its maximum over the sweep,
    by linear interpolation, and their difference; None where the curve never reaches 0.1."""
    top = mean.max()
    if top < 0.1:
        return None
    def cross(level):
        i = int(np.argmax(mean >= level))
        if i == 0:
            return DEGREES[0]
        x0, x1, y0, y1 = DEGREES[i - 1], DEGREES[i], mean[i - 1], mean[i]
        return x0 + (level - y0) * (x1 - x0) / (y1 - y0)
    lo, hi = cross(0.1 * top), cross(0.9 * top)
    return lo, hi, hi - lo, top

print("model            N      d at 10%   d at 90%   width   top fraction")
for model in MODELS:
    for n in SIZES:
        w = width(curves[(n, model)].mean(axis=1))
        print(f"{model:15s}{n:6d}   " + ("never reaches 0.1" if w is None else f"{w[0]:8.2f}   {w[1]:8.2f}   {w[2]:5.2f}   {w[3]:.3f}"))

model            N      d at 10%   d at 90%   width   top fraction
pure              100       2.00       3.14    1.14   1.000
pure             1000       2.30       2.98    0.68   0.997
pure            10000       2.32       3.57    1.26   0.998
plan budget       100       2.25       3.24    0.98   0.996
plan budget      1000       2.53       3.19    0.66   0.998
plan budget     10000       2.54       3.09    0.55   0.997
true budget 0.2   100   never reaches 0.1
true budget 0.2  1000   never reaches 0.1
true budget 0.2 10000   never reaches 0.1
true budget 0.6   100       1.27       3.40    2.13   0.995
true budget 0.6  1000       1.66       3.41    1.75   0.993
true budget 0.6 10000       1.78       3.10    1.32   0.993
fan-out           100       2.81       4.47    1.65   0.998
fan-out          1000       4.00       4.25    0.25   0.996
fan-out         10000       4.03       4.24    0.21   0.995
inhibition        100       4.26       5.34    1.08   0.886
inhibition       1000      